# Diabetes Prediction Model (Clean & Optimized)

This notebook builds a machine learning model to predict:

➡️ `diabetes`

## What this notebook does:

1. Loads dataset
2. Removes duplicates
3. Prevents data leakage
4. Trains multiple models
5. Hyperparameter tuning
6. Threshold optimization
7. Saves best model
8. Runs predictions

This is a clean, production-style pipeline.

### Run only if needed
#### !pip install pandas numpy scikit-learn scipy joblib matplotlib

In [33]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.stats import randint, loguniform

from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

In [37]:
RANDOM_STATE = 42
TARGET = "diabetes"   # 👈 THIS IS THE KEY CHANGE

DATA_PATH = Path("heart_failure_clinical_records.csv")

OUTPUT_DIR = Path("outputs_diabetes")
MODEL_DIR = Path("models_diabetes")

OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

N_ITER_PER_MODEL = 25
CV_SPLITS = 5
SCORING = "average_precision"

print("Target:", TARGET)

Target: diabetes


In [38]:
df_raw = pd.read_csv(DATA_PATH)

print("Shape:", df_raw.shape)
display(df_raw.head())

Shape: (5000, 13)


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,55.0,0,748,0,45,0,263358.03,1.3,137,1,1,88,0
1,65.0,0,56,0,25,0,305000.00,5.0,130,1,0,207,0
2,45.0,0,582,1,38,0,319000.00,0.9,140,0,0,244,0
3,60.0,1,754,1,40,1,328000.00,1.2,126,1,0,90,0
4,95.0,1,582,0,30,0,461000.00,2.0,132,1,0,50,1


In [39]:
df = df_raw.drop_duplicates().reset_index(drop=True)

print("After removing duplicates:", df.shape)

print("\nTarget distribution:")
display(df[TARGET].value_counts())
display(df[TARGET].value_counts(normalize=True))

After removing duplicates: (1320, 13)

Target distribution:


diabetes
0    730
1    590
Name: count, dtype: int64

diabetes
0    0.55303
1    0.44697
Name: proportion, dtype: float64

In [40]:
# 🚫 Remove target from features
feature_columns = [col for col in df.columns if col != TARGET]

X = df[feature_columns]
y = df[TARGET]

print("Features used:", feature_columns)

Features used: ['age', 'anaemia', 'creatinine_phosphokinase', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time', 'DEATH_EVENT']


In [41]:
def get_probability(model, X):
    return model.predict_proba(X)[:, 1]


def find_best_threshold(y_true, probabilities):
    rows = []

    for t in np.linspace(0.05, 0.95, 181):
        pred = (probabilities >= t).astype(int)

        rows.append({
            "threshold": t,
            "f1": f1_score(y_true, pred),
            "precision": precision_score(y_true, pred),
            "recall": recall_score(y_true, pred),
        })

    df_t = pd.DataFrame(rows).sort_values("f1", ascending=False)
    return float(df_t.iloc[0]["threshold"]), df_t


def evaluate(model, X_test, y_test, threshold):
    probs = get_probability(model, X_test)
    preds = (probs >= threshold).astype(int)

    return {
        "accuracy": accuracy_score(y_test, preds),
        "f1": f1_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, probs),
        "pr_auc": average_precision_score(y_test, probs),
    }, preds

In [42]:
def get_models():
    return {
        "logistic": {
            "pipeline": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler()),
                ("model", LogisticRegression(max_iter=5000))
            ]),
            "params": {
                "model__C": loguniform(0.01, 10),
                "model__class_weight": [None, "balanced"]
            }
        },

        "random_forest": {
            "pipeline": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", RandomForestClassifier(n_jobs=-1))
            ]),
            "params": {
                "model__n_estimators": randint(200, 600),
                "model__max_depth": [None, 4, 6, 10],
                "model__class_weight": [None, "balanced"]
            }
        },

        "extra_trees": {
            "pipeline": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", ExtraTreesClassifier(n_jobs=-1))
            ]),
            "params": {
                "model__n_estimators": randint(200, 600),
                "model__max_depth": [None, 4, 6, 10],
            }
        },

        "hgb": {
            "pipeline": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", HistGradientBoostingClassifier())
            ]),
            "params": {
                "model__learning_rate": loguniform(0.01, 0.2),
                "model__max_iter": randint(100, 400),
            }
        }
    }